In [ ]:
# Import necessary libraries
import os
from vllm import LLM, SamplingParams
from vllm.steer_vectors.request import SteerVectorRequest

# Set environment variables
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Initialize LLM
llm = LLM(
    model="Qwen/Qwen3-0.6B",
    enable_steer_vector=True,
    enforce_eager=True,
    tensor_parallel_size=1,
    enable_chunked_prefill=False
)

# Define prompts that should elicit an explicitly happy tone (steer targets)
happy_prompts = [
    "Answer the following question with an upbeat, joyful tone: Give three tips for staying healthy.",
    "Respond enthusiastically and happily: What are the three primary colors?",
    "Cheerfully describe the structure of an atom so it sounds exciting."
]

# Define neutral prompts that keep a matter-of-fact tone (base targets)
neutral_prompts = [
    "Give three tips for staying healthy in a straightforward tone.",
    "What are the three primary colors? Answer plainly without added emotion.",
    "Describe the structure of an atom in an objective tone."
]

# Create properly formatted prompts for both happy and neutral queries
# The format follows the model's expected chat template
texts = [f"<|im_start|>user\n{x}<|im_end|>\n<|im_start|>assistant\n" for x in happy_prompts + neutral_prompts]

# Generate responses for all queries
# This will validate that the model correctly refuses harmful queries
answers = llm.generate(
    texts,
    SamplingParams(
        temperature=0,        # Deterministic generation
        max_tokens=128,       # Limit response length
        skip_special_tokens=False,  # Keep special tokens intact
    ),
)
answers = [answer.outputs[0].text for answer in answers]

In [ ]:
print(answers)

In [ ]:
from transformers import AutoTokenizer
# Examine how the tokenizer processes our prompt
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
tokens = tokenizer.tokenize(texts[0], add_special_tokens=True)
print(tokens)

In [ ]:
import gc
del llm
gc.collect()

In [ ]:

# According to the original paper, we only need to extract the positions of:
# '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ'
# These correspond to token positions -1, -2, -3, -4
# Import hidden states module to extract model activations
import easysteer.hidden_states as hs

# Create a new LLM instance in reward mode
# Note: This allows us to extract hidden states rather than generating text
llm = LLM(
    model="Qwen/Qwen3-0.6B",
    task="embed", 
    tensor_parallel_size=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False
)

# Extract hidden states for all tokens in the QA pairs
all_hidden_states, outputs = hs.get_all_hidden_states(llm, texts)

In [ ]:
from easysteer.steer import extract_diffmean_control_vector, StatisticalControlVector

# Extract happy-vs-neutral direction vectors at several final token positions
# positive_indices: happy_prompts (indices 0,1,2); negative_indices: neutral_prompts (3,4,5)

# Position -1 (last token)
control_vector = extract_diffmean_control_vector(
    all_hidden_states=all_hidden_states,  # 3D list [sample][layer][token]
    positive_indices=[0, 1, 2],          # Happy examples
    negative_indices=[3, 4, 5],          # Neutral examples
    model_type="qwen3",                 # Model type (Qwen3 family)
    token_pos=-1,                        # Target the last token position
    normalize=True                       # Normalize the resulting vector
)
os.makedirs("../vectors/emotion_vector", exist_ok=True)
control_vector.export_gguf("../vectors/emotion_vector/happy-diffmean-1.gguf")  # Save vector to file

# Position -2 (second-to-last token)
# control_vector = extract_diffmean_control_vector(
#     all_hidden_states=all_hidden_states,
#     positive_indices=[0, 1, 2],
#     negative_indices=[3, 4, 5],
#     model_type="qwen3",
#     token_pos=-2,
#     normalize=True
# )
# control_vector.export_gguf("happy-diffmean-2.gguf")

# # Position -3 (third-to-last token)
# control_vector = extract_diffmean_control_vector(
#     all_hidden_states=all_hidden_states,
#     positive_indices=[0, 1, 2],
#     negative_indices=[3, 4, 5],
#     model_type="qwen3",
#     token_pos=-3,
#     normalize=True
# )
# control_vector.export_gguf("happy-diffmean-3.gguf")

# # Position -4 (fourth-to-last token)
# control_vector = extract_diffmean_control_vector(
#     all_hidden_states=all_hidden_states,
#     positive_indices=[0, 1, 2],
#     negative_indices=[3, 4, 5],
#     model_type="qwen3",
#     token_pos=-4,
#     normalize=True
# )
# control_vector.export_gguf("happy-diffmean-4.gguf")

ERROR 12-01 09:10:57 [core_client.py:616] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
